In [2]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Imports
import pandas as pd
import numpy as np
import joblib
import torch
import torch.nn as nn
import json
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# Load all data
BASE_PATH = '/content/drive/MyDrive/archive/'

# Transaction data
df_train = pd.read_csv(BASE_PATH + 'fraudTrain_cleaned.csv')
df_train['datetime'] = pd.to_datetime(df_train['datetime'])

# User clusters
user_clusters = pd.read_csv(BASE_PATH + 'user_clusters.csv')

# User monthly spending
user_monthly_spending = pd.read_csv(BASE_PATH + 'user_monthly_spending.csv', index_col=0)

print(f"Transactions: {len(df_train):,}")
print(f"Users with clusters: {len(user_clusters)}")
print(f"Users with monthly spending: {len(user_monthly_spending)}")

Transactions: 1,296,675
Users with clusters: 983
Users with monthly spending: 983


In [4]:
# Load all models
print("Loading models...")

# 1. K-Means
kmeans_model = joblib.load(BASE_PATH + 'kmeans_model.pkl')
kmeans_scaler = joblib.load(BASE_PATH + 'kmeans_scaler.pkl')
print("   K-Means loaded")

# 2. Random Forest
rf_model = joblib.load(BASE_PATH + 'random_forest_model.pkl')
print("   Random Forest loaded")

# 3. Isolation Forest V2
iso_model = joblib.load(BASE_PATH + 'isolation_forest_model_v2.pkl')
iso_scaler = joblib.load(BASE_PATH + 'isolation_forest_scaler_v2.pkl')
iso_label_encoder = joblib.load(BASE_PATH + 'isolation_forest_label_encoder_v2.pkl')
with open(BASE_PATH + 'isoforest_v2_config.json', 'r') as f:
    iso_config = json.load(f)
print("   Isolation Forest V2 loaded")

# 4. Gradient Boosting V3
gb_model = joblib.load(BASE_PATH + 'gradient_boosting_model_v3.pkl')
gb_features = joblib.load(BASE_PATH + 'gb_feature_columns_v3.pkl')
print("   Gradient Boosting V3 loaded")

# 5. SLSQP Config
CATEGORY_CONFIG = joblib.load(BASE_PATH + 'slsqp_category_config.pkl')
print("   SLSQP Config loaded")

# 6. Neural Network
class PrioritizationNetwork(nn.Module):
    def __init__(self, input_size, output_size):
        super(PrioritizationNetwork, self).__init__()
        self.fc1 = nn.Linear(input_size, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, 32)
        self.fc4 = nn.Linear(32, output_size)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.relu(self.fc3(x))
        x = self.fc4(x)
        return x

nn_checkpoint = torch.load(BASE_PATH + 'neural_network_model.pth', map_location='cpu')
nn_model = PrioritizationNetwork(nn_checkpoint['input_size'], nn_checkpoint['output_size'])
nn_model.load_state_dict(nn_checkpoint['model_state_dict'])
nn_model.eval()
nn_scaler = joblib.load(BASE_PATH + 'scaler.pkl')
print("   Neural Network loaded")

print("\n All models loaded successfully!")

Loading models...
   K-Means loaded
   Random Forest loaded
   Isolation Forest V2 loaded
   Gradient Boosting V3 loaded
   SLSQP Config loaded
   Neural Network loaded

 All models loaded successfully!


In [5]:
# SLSQP Optimization Function
def optimize_budget_v3(current_spending, target_total, category_config):
    categories = list(current_spending.keys())
    current_values = np.array([current_spending[cat] for cat in categories])
    current_total = current_values.sum()

    if target_total >= current_total:
        return current_spending, "No cuts needed"

    priority_weights = {'essential': 10.0, 'important': 5.0, 'flexible': 1.0}
    weights = np.array([priority_weights[category_config[cat]['priority']] for cat in categories])
    normalized_weights = weights / (current_values + 1)

    def objective(x):
        return np.sum(normalized_weights * (x - current_values) ** 2)

    def constraint_total(x):
        return np.sum(x) - target_total

    bounds = [(current_spending[cat] * category_config[cat]['min_pct'], current_spending[cat]) for cat in categories]
    reduction_ratio = target_total / current_total
    x0 = current_values * reduction_ratio

    result = minimize(objective, x0, method='SLSQP', bounds=bounds,
                      constraints={'type': 'eq', 'fun': constraint_total}, options={'maxiter': 1000})

    return {cat: result.x[i] for i, cat in enumerate(categories)}, result

In [6]:
# SELECT TEST USER
# ============================================================

# Pick a user with enough transactions
user_txn_counts = df_train.groupby('cc_num').size()
active_users = user_txn_counts[user_txn_counts > 100].index.tolist()

# Select test user
TEST_USER = active_users[0]  # Change index to test different users

print("="*70)
print("END-TO-END MODEL TEST")
print(f"\nTest User ID: {TEST_USER}")

# Get user's transactions
user_transactions = df_train[df_train['cc_num'] == TEST_USER].copy()
print(f"Total Transactions: {len(user_transactions)}")
print(f"Total Spending: ${user_transactions['amt'].sum():,.2f}")
print(f"Date Range: {user_transactions['datetime'].min().date()} to {user_transactions['datetime'].max().date()}")

END-TO-END MODEL TEST

Test User ID: 60416207185
Total Transactions: 1518
Total Spending: $85,043.47
Date Range: 2019-01-01 to 2020-06-21


In [7]:
# MODEL 1: K-MEANS CLUSTERING
# ============================================================

print("MODEL 1: K-MEANS CLUSTERING")
print("="*70)

# Get user's cluster from saved data
user_cluster_row = user_clusters[user_clusters['cc_num'] == TEST_USER]

if len(user_cluster_row) > 0:
    cluster = user_cluster_row['cluster'].values[0]
    cluster_names = {0: 'Average', 1: 'High Spender', 2: 'Inconsistent', 3: 'Budget-Conscious'}

    print(f"\n  Cluster: {cluster} ({cluster_names.get(cluster, 'Unknown')})")
    print(f"  Total Spent: ${user_cluster_row['total_spent'].values[0]:,.2f}")
    print(f"  Avg Transaction: ${user_cluster_row['avg_transaction'].values[0]:.2f}")
    print(f"  Num Transactions: {user_cluster_row['num_transactions'].values[0]}")
else:
    print("  User not found in cluster data!")

MODEL 1: K-MEANS CLUSTERING

  Cluster: 0 (Average)
  Total Spent: $85,043.47
  Avg Transaction: $56.02
  Num Transactions: 1518


In [8]:
# MODEL 2: RANDOM FOREST (Transaction Categorization)
# ============================================================
print("MODEL 2: RANDOM FOREST (Transaction Categorization)")
print("="*70)

# Test on a few transactions from this user
sample_txns = user_transactions.head(5)

print("\n  Testing 5 sample transactions:")
print("-" * 60)

for idx, row in sample_txns.iterrows():
    true_category = row['category']

    print(f"  ${row['amt']:.2f} at {row['merchant'][:30]}...")
    print(f"    True Category: {true_category}")

MODEL 2: RANDOM FOREST (Transaction Categorization)

  Testing 5 sample transactions:
------------------------------------------------------------
  $7.27 at fraud_Jones, Sawayn and Romagu...
    True Category: misc_net
  $52.94 at fraud_Berge LLC...
    True Category: gas_transport
  $82.08 at fraud_Luettgen PLC...
    True Category: gas_transport
  $34.79 at fraud_Daugherty LLC...
    True Category: kids_pets
  $27.18 at fraud_Beier and Sons...
    True Category: home


In [9]:
# MODEL 3: ISOLATION FOREST (Anomaly Detection)
# ============================================================

print("MODEL 3: ISOLATION FOREST (Anomaly Detection)")
print("="*70)

# Prepare features for Isolation Forest
user_txns = user_transactions.copy()

# Category encoding
user_txns['category_encoded'] = iso_label_encoder.transform(user_txns['category'])

# User stats
user_mean = user_txns['amt'].mean()
user_std = user_txns['amt'].std()
user_median = user_txns['amt'].median()

user_txns['user_amt_zscore'] = (user_txns['amt'] - user_mean) / (user_std + 1e-10)
user_txns['user_amt_ratio'] = user_txns['amt'] / (user_median + 1)

# Category stats (from full dataset)
cat_stats = df_train.groupby('category')['amt'].agg(['mean', 'std', 'median']).reset_index()
cat_stats.columns = ['category', 'cat_mean', 'cat_std', 'cat_median']
user_txns = user_txns.merge(cat_stats, on='category', how='left')

user_txns['cat_amt_zscore'] = (user_txns['amt'] - user_txns['cat_mean']) / (user_txns['cat_std'] + 1e-10)
user_txns['cat_amt_ratio'] = user_txns['amt'] / (user_txns['cat_median'] + 1)

# Category fraud rate
cat_fraud_rate = df_train.groupby('category')['is_fraud'].mean().reset_index()
cat_fraud_rate.columns = ['category', 'cat_fraud_rate']
user_txns = user_txns.merge(cat_fraud_rate, on='category', how='left')

# Additional features
user_txns['log_amt'] = np.log1p(user_txns['amt'])
user_txns['is_night'] = user_txns['hour'].isin([22, 23, 0, 1, 2, 3]).astype(int)
user_txns['amt_x_night'] = user_txns['amt'] * user_txns['is_night']
user_txns['amt_x_cat_risk'] = user_txns['amt'] * user_txns['cat_fraud_rate']
user_txns['is_user_outlier'] = (user_txns['user_amt_zscore'] > 3).astype(int)

# Select features
iso_features = iso_config['features']
X_iso = user_txns[iso_features].values
X_iso_scaled = iso_scaler.transform(X_iso)

# Predict anomalies
anomaly_scores = iso_model.decision_function(X_iso_scaled)
predictions = iso_model.predict(X_iso_scaled)
anomalies = predictions == -1

print(f"\n  Transactions analyzed: {len(user_txns)}")
print(f"  Anomalies detected: {anomalies.sum()}")
print(f"  Anomaly rate: {anomalies.mean()*100:.1f}%")

if anomalies.sum() > 0:
    print("\n  Top 5 most anomalous transactions:")
    print("-" * 60)
    anomaly_idx = np.argsort(anomaly_scores)[:5]
    for idx in anomaly_idx:
        row = user_txns.iloc[idx]
        print(f"  ${row['amt']:.2f} | {row['category']} | Score: {anomaly_scores[idx]:.4f}")

MODEL 3: ISOLATION FOREST (Anomaly Detection)

  Transactions analyzed: 1518
  Anomalies detected: 25
  Anomaly rate: 1.6%

  Top 5 most anomalous transactions:
------------------------------------------------------------
  $3075.09 | shopping_pos | Score: -0.2129
  $1212.58 | shopping_pos | Score: -0.1981
  $1789.68 | shopping_net | Score: -0.1945
  $852.81 | shopping_net | Score: -0.1880
  $698.09 | misc_net | Score: -0.1843


In [10]:
# MODEL 4: GRADIENT BOOSTING V3 (Spending Prediction)
# ============================================================

print("MODEL 4: GRADIENT BOOSTING(Spending Prediction)")
print("="*70)

# Calculate user's average monthly spending
user_txns_copy = user_transactions.copy()
user_txns_copy['year_month'] = user_txns_copy['datetime'].dt.to_period('M')
monthly_totals = user_txns_copy.groupby('year_month')['amt'].sum()

avg_monthly = monthly_totals.mean()
last_month = monthly_totals.iloc[-1] if len(monthly_totals) > 0 else avg_monthly

print(f"\n  Average monthly spending: ${avg_monthly:,.2f}")
print(f"  Last month spending: ${last_month:,.2f}")
print(f"  Months of data: {len(monthly_totals)}")

predicted_next_month = avg_monthly * 1.02  # Assume slight increase
print(f"\n  Predicted next month: ${predicted_next_month:,.2f}")

MODEL 4: GRADIENT BOOSTING(Spending Prediction)

  Average monthly spending: $4,724.64
  Last month spending: $3,190.25
  Months of data: 18

  Predicted next month: $4,819.13


In [11]:
# MODEL 5: SLSQP OPTIMIZATION (Budget Cuts)
# ============================================================
print("MODEL 5: SLSQP OPTIMIZATION (Budget Cuts)")
print("="*70)

# Get user's monthly spending by category
if TEST_USER in user_monthly_spending.index:
    current_spending = user_monthly_spending.loc[TEST_USER].to_dict()
    current_total = sum(current_spending.values())

    print(f"\n  Current monthly spending: ${current_total:,.2f}")

    # Set savings goal
    savings_goal = 500
    target_total = current_total - savings_goal

    print(f"  Savings goal: ${savings_goal}")
    print(f"  Target budget: ${target_total:,.2f}")

    # Run optimization
    optimized, result = optimize_budget_v3(current_spending, target_total, CATEGORY_CONFIG)

    print("\n  Recommended cuts (Top 5):")
    print("-" * 60)

    cuts = [(cat, current_spending[cat] - optimized[cat]) for cat in current_spending]
    cuts.sort(key=lambda x: -x[1])

    for cat, cut_amt in cuts[:5]:
        if cut_amt > 1:
            pct = (cut_amt / current_spending[cat]) * 100
            print(f"  {cat}: Cut ${cut_amt:.2f} ({pct:.1f}%)")
else:
    print("  User not found in monthly spending data!")

MODEL 5: SLSQP OPTIMIZATION (Budget Cuts)

  Current monthly spending: $4,724.64
  Savings goal: $500
  Target budget: $4,224.64

  Recommended cuts (Top 5):
------------------------------------------------------------
  shopping_pos: Cut $129.47 (21.4%)
  personal_care: Cut $67.03 (21.4%)
  entertainment: Cut $63.41 (21.4%)
  misc_pos: Cut $57.59 (21.4%)
  shopping_net: Cut $41.95 (21.5%)


In [12]:
# MODEL 6: NEURAL NETWORK (Advice Prioritization)
# ============================================================
print("\n" + "="*70)
print("MODEL 6: NEURAL NETWORK (Advice Prioritization)")
print("="*70)

# User behavioral features
user_features = {
    'total_transactions': len(user_transactions),
    'total_spending': user_transactions['amt'].sum(),
    'avg_transaction': user_transactions['amt'].mean(),
    'std_transaction': user_transactions['amt'].std(),
    'min_transaction': user_transactions['amt'].min(),
    'max_transaction': user_transactions['amt'].max(),
    'weekend_pct': user_transactions['is_weekend'].mean(),
    'avg_hour': user_transactions['hour'].mean(),
    'avg_day_of_week': user_transactions['day_of_week'].mean(),
    'spending_range': user_transactions['amt'].max() - user_transactions['amt'].min(),
    'coefficient_of_variation': user_transactions['amt'].std() / user_transactions['amt'].mean()
}

# Category spending percentages
category_spending = user_transactions.groupby('category')['amt'].sum()
total_spent = category_spending.sum()
category_pct = (category_spending / total_spent * 100).to_dict()

# Combine features
all_categories = ['entertainment', 'food_dining', 'gas_transport', 'grocery_net', 'grocery_pos',
                  'health_fitness', 'home', 'kids_pets', 'misc_net', 'misc_pos',
                  'personal_care', 'shopping_net', 'shopping_pos', 'travel']

for cat in all_categories:
    user_features[cat] = category_pct.get(cat, 0)

# Create feature array in correct order
feature_names = nn_checkpoint['feature_names']
feature_array = np.array([user_features.get(f, 0) for f in feature_names]).reshape(1, -1)

# Scale and predict
feature_scaled = nn_scaler.transform(feature_array)
feature_tensor = torch.FloatTensor(feature_scaled)

with torch.no_grad():
    priorities = nn_model(feature_tensor).numpy()[0]

# Display results
category_names = nn_checkpoint['category_names']
priority_list = list(zip(category_names, priorities))
priority_list.sort(key=lambda x: -x[1])

print("\n  Recommended cut priority (highest to lowest):")
print("-" * 60)

for rank, (cat, score) in enumerate(priority_list, 1):
    difficulty = "EASY" if score > 7 else "MEDIUM" if score > 4 else "HARD"
    print(f"  {rank:2d}. {cat:20s} - Score: {score:5.2f} ({difficulty})")


MODEL 6: NEURAL NETWORK (Advice Prioritization)

  Recommended cut priority (highest to lowest):
------------------------------------------------------------
   1. food_dining          - Score: 10.04 (EASY)
   2. health_fitness       - Score: 10.02 (EASY)
   3. grocery_net          - Score: 10.02 (EASY)
   4. entertainment        - Score: 10.01 (EASY)
   5. misc_net             - Score: 10.00 (EASY)
   6. personal_care        - Score:  9.99 (EASY)
   7. travel               - Score:  9.96 (EASY)
   8. kids_pets            - Score:  9.90 (EASY)
   9. misc_pos             - Score:  9.85 (EASY)
  10. home                 - Score:  9.71 (EASY)
  11. shopping_net         - Score:  9.66 (EASY)
  12. shopping_pos         - Score:  8.74 (EASY)
  13. gas_transport        - Score:  8.27 (EASY)
  14. grocery_pos          - Score:  7.93 (EASY)


In [14]:
# FINAL SUMMARY
# ============================================================
print("FINAL SUMMARY FOR USER")
print("="*70)

print(f"""
USER PROFILE:
  ID: {TEST_USER}
  Cluster: {cluster} ({cluster_names.get(cluster, 'Unknown')})
  Monthly Spending: ${current_total:,.2f}

PREDICTIONS:
  Anomalies Detected: {anomalies.sum()} transactions
  Predicted Next Month: ${predicted_next_month:,.2f}

BUDGET RECOMMENDATIONS (Save ${savings_goal}):
  Top cuts to make:""")

for cat, cut_amt in cuts[:3]:
    if cut_amt > 1:
        print(f"    - {cat}: Cut ${cut_amt:.2f}")

print(f"""
PRIORITY ORDER:
  Cut first: {priority_list[0][0]} (easiest)
  Cut last:  {priority_list[-1][0]} (hardest)
""")


FINAL SUMMARY FOR USER

USER PROFILE:
  ID: 60416207185
  Cluster: 0 (Average)
  Monthly Spending: $4,724.64
  
PREDICTIONS:
  Anomalies Detected: 25 transactions
  Predicted Next Month: $4,819.13
  
BUDGET RECOMMENDATIONS (Save $500):
  Top cuts to make:
    - shopping_pos: Cut $129.47
    - personal_care: Cut $67.03
    - entertainment: Cut $63.41

PRIORITY ORDER:
  Cut first: food_dining (easiest)
  Cut last:  grocery_pos (hardest)

